In [ ]:
import sys
sys.path.extend([".", "fashion_recsys"])

import numpy as np
from IPython.display import display
from config import *
import data, features, scratch, analysis, serve

In [ ]:
cat = data.load_catalogue()
small = data.load_small(cat.paths)

In [ ]:
# pretrained resnet-50 and vit-b/16, plus the cnn autoencoder trained from scratch
emb, cost = features.pretrained_embeddings(cat.paths)
emb["CNN Autoencoder (scratch)"], cost["CNN Autoencoder (scratch)"] = scratch.train_autoencoder(small, cat.train_idx)

In [ ]:
# our own vit: first row is the baseline, every other row changes one thing
vit_df, vit_embs, vit_cost = analysis.vit_ablation(small, cat)
base = vit_df.index[0]
emb["ViT (scratch)"], cost["ViT (scratch)"] = vit_embs[base], vit_cost[base]
emb["Random"] = np.random.default_rng(SEED).standard_normal((len(cat.df), 64))
vit_df.round(3)

In [ ]:
results, best = analysis.compare(emb, cat)
print("best:", best)
analysis.plot_comparison(results)
results[["Dim", f"P@{K}", f"MAP@{K}", "MAP_lo", "MAP_hi", f"R@{K}"]].round(3)

In [ ]:
analysis.paired(emb, cat, "ViT-B/16 (CLS)", "ResNet-50 (avg pool)")
analysis.paired(emb, cat, "ViT-B/16 (CLS)", "ViT-B/16 (mean patch)")
analysis.paired(emb, cat, "ResNet-50 (GeM pool)", "ResNet-50 (avg pool)")

In [ ]:
pca_df = analysis.pca_ablation(emb[best], cat)
print("cosine vs euclidean (resnet-50):", analysis.metric_ablation(emb["ResNet-50 (avg pool)"], cat))
pca_df.round(3)

In [ ]:
analysis.relevance_table(emb, [best, "ResNet-50 (avg pool)", "CNN Autoencoder (scratch)", "ViT (scratch)"], cat).round(3)

In [ ]:
cost_df = analysis.cost_table(emb, cost, results)
cost_df.round(3)

In [ ]:
worst_classes, confusions, worst_q = analysis.error_analysis(emb[best], cat)
display(worst_classes); display(confusions)

analysis.show(emb[best], cat, np.random.default_rng(SEED).choice(cat.test_idx, 4, replace=False), fname="examples.png")
analysis.show(emb[best], cat, worst_q, fname="failures.png")

In [ ]:
serve.save_index(emb[best], cat)     # then: uvicorn app:app